In [1]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/beshoyshohdy/meta123/meta.csv")

In [2]:
df['diagnosis'] = df['diagnosis'].replace({'ak':'akiec'})

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['label'] = le.fit_transform(df['diagnosis'])

In [4]:
df['label'].value_counts()

label
5    9203
4    5625
1    3830
2    3709
0    1186
6     628
7     395
3     354
Name: count, dtype: int64

In [5]:
df['diagnosis'].value_counts()

diagnosis
nv       9203
mel      5625
bcc      3830
bkl      3709
akiec    1186
scc       628
vasc      395
df        354
Name: count, dtype: int64

In [6]:
class_names = [
    "akiec",  # 0
    "bcc",    # 1
    "bkl",    # 2
    "df",     # 3
    "mel",    # 4
    "nv",     # 5
    "scc",    # 6
    "vasc"    # 7
]

In [7]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 17451
Validation: 3739
Test: 3740


In [8]:
train_df.head()

,image_path,diagnosis,age,sex,anatom_site_general_challenge,split,label
4658,/kaggle/input/datasets/beshoyshohdy/skin-data/...,bcc,45.0,female,anterior torso,train,1
21466,/kaggle/input/datasets/beshoyshohdy/skin-data/...,nv,65.0,male,upper extremity,test,5
19064,/kaggle/input/datasets/beshoyshohdy/skin-data/...,nv,70.0,male,upper extremity,train,5
4709,/kaggle/input/datasets/beshoyshohdy/skin-data/...,akiec,70.0,female,head/neck,train,0
20882,/kaggle/input/datasets/beshoyshohdy/skin-data/...,nv,75.0,female,upper extremity,val,5


In [9]:
import os
import tensorflow as tf

IMG_SIZE = (300, 300)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label


def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)

    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k=k)

    image = tf.image.resize_with_crop_or_pad(image, 330, 330)
    image = tf.image.random_crop(
        image, size=[300, 300, 3]
    )

    image = tf.image.random_brightness(image, max_delta=20.0)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.random_saturation(image, 0.85, 1.15)
    image = tf.clip_by_value(image, 0.0, 255.0)

    return image, label


def create_dataset(dataframe, training=False):
    paths = dataframe["image_path"].to_numpy()
    labels = dataframe["label"].to_numpy().astype("int32")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    
    if training:
        ds = ds.shuffle(
            min(len(paths), 5000),
            seed=42,
            reshuffle_each_iteration=True
        )
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

print("Datasets created successfully.")

I0000 00:00:1793100828.042242     752 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793100828.047663     752 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Datasets created successfully.


In [10]:
to stage 3

SyntaxError: invalid syntax (1619925370.py, line 1)

# model

# stage 1

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB6

# Create a fresh strategy after restarting
strategy = tf.distribute.MirroredStrategy()
print("Replicas:", strategy.num_replicas_in_sync)

with strategy.scope():
    base_model = EfficientNetB6(
        include_top=False,
        weights="imagenet",
        input_shape=(300, 300, 3),
        pooling="avg"
    )
    base_model.trainable = False

    inputs = layers.Input(shape=(300, 300, 3))
    x = base_model(inputs, training=False)
    
  #  x = layers.Dense(1024)(x)
  #  x = layers.BatchNormalization()(x)
  #  x = layers.Activation("relu")(x)
  #  x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(512)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(64)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2)(x)
    
    outputs = layers.Dense(9, activation="softmax")(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(train_df["label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].to_numpy()
)

class_weights = dict(zip(classes, class_weights))

print(class_weights)

In [ ]:

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)


# stage 2

In [ ]:
import os
import tensorflow as tf

IMG_SIZE = (300, 300)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label


def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)

    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k=k)

    image = tf.image.resize_with_crop_or_pad(image, 330, 330)
    image = tf.image.random_crop(
        image, size=[300, 300, 3]
    )

    image = tf.image.random_brightness(image, max_delta=20.0)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.random_saturation(image, 0.85, 1.15)
    image = tf.clip_by_value(image, 0.0, 255.0)

    return image, label


def create_dataset(dataframe, training=False):
    paths = dataframe["image_path"].to_numpy()
    labels = dataframe["label"].to_numpy().astype("int32")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    
    if training:
        ds = ds.shuffle(
            min(len(paths), 5000),
            seed=42,
            reshuffle_each_iteration=True
        )
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

print("Datasets created successfully.")

In [ ]:
model.load_weights("best_model.keras")

with strategy.scope():
    for layer in base_model.layers:
        layer.trainable = layer.name.startswith(("block6", "block7", "top_"))
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(
        optimizer=tf.keras.optimizers.Adam(5e-5),
        loss=model.loss,          # keep the same loss you used in stage 1
        metrics=["accuracy"],
    )

model.summary()

In [ ]:

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-8,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model_stage_2.keras",
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)


In [ ]:
class_names = [
    "ak",      # 0
    "akiec",   # 1
    "bcc",     # 2
    "bkl",     # 3
    "df",      # 4
    "mel",     # 5
    "nv",      # 6
    "scc",     # 7
    "vasc"     # 8
]

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

# Predict probabilities for each class
y_prob = model.predict(test_ds)

# Predicted class indices
y_pred = np.argmax(y_prob, axis=1)

# Extract integer labels from test_ds
y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
]).astype(np.int32)

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)

print("\nTest Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(class_names))
))


# stage 3

In [ ]:
!kaggle kernels output beshoyshohdy/skin-classifier -p /kaggle/working

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    model = tf.keras.models.load_model("best_model_stage_3.keras")
    base_model = [l for l in model.layers if isinstance(l, tf.keras.Model)][0]

    base_model.trainable = True
    for layer in base_model.layers:
        layer.trainable = layer.name.startswith(("block5", "block6", "block7", "top_"))
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(5e-6),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(train_df["label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].to_numpy()
)

class_weights = dict(zip(classes, class_weights))

print(class_weights)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-8,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model_stage_3.keras",
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

# Predict probabilities for each class
y_prob = model.predict(test_ds)

# Predicted class indices
y_pred = np.argmax(y_prob, axis=1)

# Extract integer labels from test_ds
y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
]).astype(np.int32)

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)

print("\nTest Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(class_names))
))


# stage 4

In [ ]:
!kaggle kernels output beshoyshohdy/skin-classifier -p /kaggle/working

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    old_model = tf.keras.models.load_model("best_model_stage_3.keras")

    x = old_model.layers[-2].output                  
    new_out = layers.Dense(8, activation="softmax", name="out_8")(x)
    model = Model(old_model.input, new_out)

    # freeze everything except the new last layer
    for l in model.layers[:-1]:
        l.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )


model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(train_df["label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].to_numpy()
)

class_weights = dict(zip(classes, class_weights))

print(class_weights)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model_stage_4.keras",
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

# Predict probabilities for each class
y_prob = model.predict(test_ds)

# Predicted class indices
y_pred = np.argmax(y_prob, axis=1)

# Extract integer labels from test_ds
y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
]).astype(np.int32)

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)

print("\nTest Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(class_names))
))


# final model

In [ ]:
!kaggle kernels output beshoyshohdy/skin-classifier -p /kaggle/working

In [11]:
import tensorflow as tf
from tensorflow.keras import layers, Model

strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    model = tf.keras.models.load_model("best_model_stage_4.keras")
    base_model = [l for l in model.layers if isinstance(l, tf.keras.Model)][0]

    base_model.trainable = True
    for layer in base_model.layers:
        layer.trainable = layer.name.startswith(("top_"))
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )


model.summary()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb6 (Functional)     │ (None, 2304)           │    40,960,143 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ out_8 (Dense)                   │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,291,927 (161.33 MB)

 Trainable params: 1,327,624 (5.06 MB)

 Non-trainable params: 40,964,303 (156.27 MB)

In [12]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(train_df["label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].to_numpy()
)

class_weights = dict(zip(classes, class_weights))

print(class_weights)

{np.int64(0): np.float64(2.62816265060241), np.int64(1): np.float64(0.8136422976501305), np.int64(2): np.float64(0.8402831278890601), np.int64(3): np.float64(8.79586693548387), np.int64(4): np.float64(0.5540703581407163), np.int64(5): np.float64(0.33861766532132875), np.int64(6): np.float64(4.957670454545455), np.int64(7): np.float64(7.875)}


In [13]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model_final.keras",
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

E0000 00:00:1793100896.414362     752 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/efficientnetb6_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


2182/2182 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.8981 - loss: 0.2691INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

Epoch 1: val_loss improved from None to 0.50081, saving model to best_model_final.keras

Epoch 1: finished saving model to best_model_final.keras
2182/2182 ━━━━━━━━━━━━━━━━━━━━ 367s 147ms/step - accuracy: 0.9046 - loss: 0.2451 - val_accuracy: 0.8550 - val_loss: 0.5008 - learning_rate: 1.0000e-05
Epoch 2/30
2182/2182 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.8995 - loss: 0.2472
Epoch 2: val_loss improved from 0.50081 to 0.49141, saving model to best_model_final.keras

Epoch 2: finished saving model to best_model_final.keras
2182/2182 ━━━━━━━━━━━━━━━━━━━━ 312s 139ms/step - accuracy: 0.9030 - loss: 0.2456 - val_accuracy: 0.8545 - val_loss: 0.4914 - learning_rate: 1.0000e-05
Epoch 3/30
2182/2182 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.9030 - loss: 0.2497
Epoc

KeyboardInterrupt: 

In [14]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

# Predict probabilities for each class
y_prob = model.predict(test_ds)

# Predicted class indices
y_pred = np.argmax(y_prob, axis=1)

# Extract integer labels from test_ds
y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
]).astype(np.int32)

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)

print("\nTest Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(class_names))
))


468/468 ━━━━━━━━━━━━━━━━━━━━ 160s 315ms/step
y_true shape: (3740,)
y_pred shape: (3740,)

Test Accuracy: 0.8532085561497326

Classification Report:
              precision    recall  f1-score   support

       akiec     0.7576    0.8427    0.7979       178
         bcc     0.8557    0.8974    0.8761       575
         bkl     0.7945    0.8273    0.8106       556
          df     0.9434    0.9434    0.9434        53
         mel     0.8005    0.8270    0.8135       844
          nv     0.9285    0.8646    0.8954      1381
         scc     0.7586    0.7021    0.7293        94
        vasc     0.9194    0.9661    0.9421        59

    accuracy                         0.8532      3740
   macro avg     0.8448    0.8588    0.8510      3740
weighted avg     0.8561    0.8532    0.8539      3740


Confusion Matrix:
[[ 150    9    7    0    3    0    9    0]
 [  16  516    9    1   18    7    7    1]
 [  18   17  460    0   39   18    4    0]
 [   0    1    0   50    1    1    0    0]
 [   2   3